# Lab 01 — Naive RAG: Build an End-to-End Retrieval-Augmented Generation Pipeline

> **Companion lab for** [`02_interview_bank/01-naive-rag.md`](../02_interview_bank/01-naive-rag.md)

## What you will build

A complete Naive RAG pipeline that:
1. Chunks and embeds a small in-memory document corpus
2. Stores vectors in a local Chroma database
3. Retrieves relevant chunks at query time
4. Generates answers via a free LLM (HuggingFace Inference API or local Ollama)
5. Evaluates retrieval precision and demonstrates failure modes

## Learning objectives

- Implement the three Naive RAG stages: **Index → Retrieve → Generate**
- Compare fixed, sliding-window, and semantic chunking strategies
- Measure retrieval precision with a ground-truth table
- Observe two common Naive RAG failure modes hands-on

## Prerequisites

- Python 3.9+
- A **free** HuggingFace account and token from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)  
  *(Or install Ollama locally — see Section 4 for the alternative)*

## Estimated time: ~30 minutes

> **No paid API required.** Embeddings run locally via `sentence-transformers`. The LLM uses the free HuggingFace Inference API.

---
## 0. Install dependencies

In [ ]:
%pip install -q langchain langchain-community langchain-huggingface \
                sentence-transformers chromadb huggingface_hub

---
## Setup — HuggingFace token

Get a free token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).  
Select **Read** scope. The token is never stored — it lives only in this session.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("Paste your HuggingFace token: ")

print("Token set.")

---
## Section 1 — Build a Sample Corpus

We create 10 in-memory documents representing a fictional company's internal knowledge base.  
Each document has `source` and `topic` metadata — these let us verify which chunks were retrieved later.

> **Why this matters (Q1):** In a real Naive RAG system you would load PDFs, web pages, or database exports here. The rest of the pipeline is identical.

In [ ]:
from langchain.schema import Document

raw_docs = [
    Document(
        page_content=(
            "Acme Corp was founded in 2010 and is headquartered in Austin, Texas. "
            "The company specialises in enterprise software for supply chain management. "
            "As of 2024, Acme Corp employs over 3,000 people across 12 countries."
        ),
        metadata={"source": "company_overview.txt", "topic": "company"},
    ),
    Document(
        page_content=(
            "Acme's flagship product, SupplyIQ, uses machine learning to predict "
            "demand fluctuations. It integrates with SAP, Oracle, and custom ERP systems. "
            "Pricing starts at $2,000 per month for up to 50 users."
        ),
        metadata={"source": "product_faq.txt", "topic": "product"},
    ),
    Document(
        page_content=(
            "SupplyIQ offers a 30-day free trial. No credit card is required to start. "
            "After the trial, customers can choose from Starter, Growth, or Enterprise plans. "
            "Enterprise plans include dedicated support and SLA guarantees."
        ),
        metadata={"source": "product_faq.txt", "topic": "product"},
    ),
    Document(
        page_content=(
            "All full-time employees receive 20 days of paid annual leave per year. "
            "Leave accrues monthly at 1.67 days per month. Unused leave can be carried "
            "forward up to a maximum of 10 days into the next calendar year."
        ),
        metadata={"source": "hr_policy.txt", "topic": "hr"},
    ),
    Document(
        page_content=(
            "Acme Corp offers a hybrid work policy. Employees are required to be in the "
            "office at least 2 days per week. Remote work days must be approved by the "
            "direct manager and logged in the HR portal."
        ),
        metadata={"source": "hr_policy.txt", "topic": "hr"},
    ),
    Document(
        page_content=(
            "The engineering team uses GitHub for version control. All pull requests "
            "require at least two approvals before merging. CI pipelines run on GitHub "
            "Actions and must pass before a PR can be merged."
        ),
        metadata={"source": "engineering_guide.txt", "topic": "engineering"},
    ),
    Document(
        page_content=(
            "Production deployments happen every Tuesday and Thursday at 2 PM UTC. "
            "Emergency hotfixes can be deployed at any time with approval from the "
            "on-call engineer and the VP of Engineering."
        ),
        metadata={"source": "engineering_guide.txt", "topic": "engineering"},
    ),
    Document(
        page_content=(
            "Acme Corp's data infrastructure runs on AWS. The primary database is "
            "PostgreSQL hosted on RDS. Data is replicated to a secondary region "
            "(us-west-2) with a recovery point objective (RPO) of 5 minutes."
        ),
        metadata={"source": "infrastructure.txt", "topic": "infrastructure"},
    ),
    Document(
        page_content=(
            "Customer support is available Monday to Friday, 9 AM to 6 PM EST. "
            "Enterprise customers have access to 24/7 priority support via a dedicated "
            "Slack channel and a named customer success manager."
        ),
        metadata={"source": "support_policy.txt", "topic": "support"},
    ),
    Document(
        page_content=(
            "Acme Corp's security policy requires all employees to complete annual "
            "cybersecurity training. Multi-factor authentication (MFA) is mandatory "
            "for all internal tools. Passwords must be rotated every 90 days."
        ),
        metadata={"source": "security_policy.txt", "topic": "security"},
    ),
]

print(f"Corpus size: {len(raw_docs)} documents")
for doc in raw_docs:
    print(f"  [{doc.metadata['topic']:14s}] {doc.metadata['source']}")

---
## Section 2 — Chunking Strategies

Chunking converts documents into smaller pieces that fit within the context window.  
The strategy you choose directly affects retrieval quality (see **Q6** in the interview bank).

We compare three approaches:

| Strategy | Description |
|---|---|
| **Fixed** | Split every N tokens with no overlap — fast but may cut sentences |
| **Sliding window** | Fixed size with overlap — reduces boundary artifacts |
| **Semantic separators** | Split on paragraph/sentence boundaries first — preserves meaning |

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter_fixed = RecursiveCharacterTextSplitter(
    chunk_size=200, chunk_overlap=0,
    separators=[" "]  # split on spaces only — ignores sentence boundaries
)

splitter_sliding = RecursiveCharacterTextSplitter(
    chunk_size=200, chunk_overlap=40
)

splitter_semantic = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ".", " "],
    chunk_size=200,
    chunk_overlap=20
)

chunks_fixed    = splitter_fixed.split_documents(raw_docs)
chunks_sliding  = splitter_sliding.split_documents(raw_docs)
chunks_semantic = splitter_semantic.split_documents(raw_docs)

print(f"Fixed chunks:    {len(chunks_fixed)}")
print(f"Sliding chunks:  {len(chunks_sliding)}")
print(f"Semantic chunks: {len(chunks_semantic)}")
print()
print("--- First fixed chunk ---")
print(repr(chunks_fixed[0].page_content))
print()
print("--- First sliding chunk ---")
print(repr(chunks_sliding[0].page_content))
print()
print("--- First semantic chunk ---")
print(repr(chunks_semantic[0].page_content))

> **Observation:** The semantic splitter keeps sentences intact. The fixed splitter may cut mid-word.  
> For the rest of this lab we use the **sliding window** strategy — a practical baseline.
> Overlap is set to 20% of chunk size, which balances context continuity and index size.

In [ ]:
# Use sliding window chunks for all subsequent steps
chunks = chunks_sliding
print(f"Using {len(chunks)} chunks for indexing.")

---
## Section 3 — Embeddings and Vector Store

We embed each chunk into a dense vector using `all-MiniLM-L6-v2` — a small, fast model that runs entirely on your CPU with no API call (see **Q3** for embedding strategy theory).

Vectors are stored in an **in-memory Chroma** database (no disk write needed for this lab).

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("Loading embedding model (first run downloads ~90 MB)...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)

vectorstore = Chroma.from_documents(chunks, embedding=embeddings)
print(f"Vector store ready — {vectorstore._collection.count()} vectors indexed.")

In [ ]:
# Preview retrieval with similarity scores
probe_query = "How many days of vacation do employees get?"
results_with_scores = vectorstore.similarity_search_with_score(probe_query, k=3)

print(f"Query: '{probe_query}'\n")
for doc, score in results_with_scores:
    print(f"  Score: {score:.4f} | Source: {doc.metadata['source']}")
    print(f"  Text:  {doc.page_content[:120]}...")
    print()

> **Note on scores:** Chroma returns L2 distance by default — lower is more similar.  
> In a real system you would normalise to cosine similarity (see **Q7** for ANN details).

---
## Section 4 — LLM Setup

### Option A — HuggingFace Inference API (default, free)

Calls Mistral-7B remotely via your free HuggingFace token.  
Rate-limited but requires no local GPU.

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    huggingfacehub_api_token=os.environ["HF_TOKEN"],
    max_new_tokens=256,
    temperature=0.1,
)

print("LLM ready: Mistral-7B-Instruct via HuggingFace Inference API")

### Option B — Ollama (local, fully offline)

If you prefer to run everything locally:

```bash
# 1. Install Ollama: https://ollama.com/download
# 2. Pull a model (choose one):
#    ollama pull mistral    (~4 GB)
#    ollama pull llama3     (~4.7 GB)
# 3. Start the server:  ollama serve
```

Then replace the cell above with:

In [ ]:
# OLLAMA ALTERNATIVE — uncomment and run instead of the HuggingFace cell above
#
# from langchain_community.llms import Ollama
#
# llm = Ollama(
#     model="mistral",   # or "llama3"
#     base_url="http://localhost:11434",
#     temperature=0.1,
# )
# print("LLM ready: Mistral via Ollama (local)")

---
## Section 5 — Retrieval Chain

We wire the retriever and LLM together using `RetrievalQA` (see **Q10** — the 50-line implementation).  
`chain_type="stuff"` concatenates all retrieved chunks into a single prompt.

In [ ]:
from langchain.chains import RetrievalQA

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
)

print("RetrievalQA chain ready.")

In [ ]:
def ask(question):
    response = qa_chain.invoke({"query": question})
    print(f"Q: {question}")
    print(f"A: {response['result'].strip()}")
    print("Sources retrieved:")
    for doc in response["source_documents"]:
        print(f"  - {doc.metadata['source']} ({doc.metadata['topic']})")
    print()

In [ ]:
ask("How many days of annual leave do employees receive?")

In [ ]:
ask("What is SupplyIQ and how much does it cost?")

In [ ]:
ask("What are the deployment days for production releases?")

---
## Section 6 — Retrieval Evaluation

We evaluate **context precision**: for each query, did the correct source document appear in the top-3 retrieved chunks?

This mirrors the metrics discussed in **Q4** of the interview bank.

In [ ]:
ground_truth = [
    {"query": "How many days of annual leave do employees receive?",  "expected_source": "hr_policy.txt"},
    {"query": "What is SupplyIQ and how much does it cost?",          "expected_source": "product_faq.txt"},
    {"query": "What are the deployment days for production releases?", "expected_source": "engineering_guide.txt"},
    {"query": "Where is Acme Corp headquartered?",                    "expected_source": "company_overview.txt"},
    {"query": "How often must passwords be rotated?",                  "expected_source": "security_policy.txt"},
]

hits = 0
print(f"{'Query':<55} {'Expected source':<30} {'Hit?'}")
print("-" * 95)

for item in ground_truth:
    retrieved = retriever.invoke(item["query"])
    retrieved_sources = {doc.metadata["source"] for doc in retrieved}
    match = item["expected_source"] in retrieved_sources
    hits += int(match)
    mark = "✓" if match else "✗"
    print(f"{item['query']:<55} {item['expected_source']:<30} {mark}")

precision = hits / len(ground_truth)
print()
print(f"Retrieval precision@3: {hits}/{len(ground_truth)} = {precision:.0%}")

---
## Section 7 — Failure Modes

Naive RAG has well-known failure patterns. We demonstrate two here (see **Q2** for the full list).

### Failure Mode 1 — Chunking Boundary Artifact

When a fact is split across a chunk boundary, no single chunk contains enough context to answer the question correctly.

In [ ]:
# Create a document where the key fact spans two sentences at a likely boundary
boundary_doc = Document(
    page_content=(
        "The annual performance review cycle begins in November. "
        "Reviews are completed by December 15th. "
        "Salary adjustments resulting from the review take effect on the first "
        "working day of the new calendar year, which is typically January 2nd or 3rd."
    ),
    metadata={"source": "hr_policy_perf.txt", "topic": "hr"},
)

# Split into very small chunks to force a boundary cut
tiny_splitter = RecursiveCharacterTextSplitter(
    chunk_size=120, chunk_overlap=0, separators=[". "]
)
boundary_chunks = tiny_splitter.split_documents([boundary_doc])

print("Chunks after aggressive split:")
for i, c in enumerate(boundary_chunks):
    print(f"  Chunk {i}: {repr(c.page_content)}")

In [ ]:
# Build a small vector store with only these split chunks
boundary_store = Chroma.from_documents(boundary_chunks, embedding=embeddings)
boundary_retriever = boundary_store.as_retriever(search_kwargs={"k": 1})

# Ask a question that requires BOTH the start and end of the fact
failure_query = "When do salary adjustments from the performance review take effect?"
retrieved = boundary_retriever.invoke(failure_query)

print(f"Query: {failure_query}\n")
print("Top retrieved chunk:")
print(f"  {repr(retrieved[0].page_content)}")
print()
print("Problem: The retrieved chunk mentions 'January 2nd or 3rd' but not WHY.")
print("The causal context (performance review → salary change) was cut off.")

> **Fix:** Use a larger chunk size or sliding window overlap so the fact and its context land in the same chunk.

### Failure Mode 2 — Semantic Mismatch (Query vs. Document Vocabulary)

Naive RAG embeds the raw query. If the query uses different vocabulary from the indexed documents, the embedding won't match — even if they mean the same thing.

In [ ]:
# The original corpus uses the phrase "paid annual leave"
# A user might ask about "vacation days" or "PTO" — same concept, different words

queries_same_intent = [
    "How many days of annual leave do employees receive?",   # exact vocabulary match
    "What is the PTO policy at Acme?",                      # acronym mismatch
    "How much vacation time do workers get each year?",     # synonym mismatch
]

hr_retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

print(f"{'Query':<55} {'Top retrieved source':<30} {'Correct?'}")
print("-" * 90)

for q in queries_same_intent:
    result = hr_retriever.invoke(q)
    top_source = result[0].metadata["source"] if result else "(none)"
    correct = top_source == "hr_policy.txt"
    mark = "✓" if correct else "✗"
    print(f"{q:<55} {top_source:<30} {mark}")

> **Observation:** The exact-match query retrieves the HR policy correctly.  
> "PTO" and "vacation time" may retrieve a less relevant chunk because `all-MiniLM-L6-v2`  
> handles synonyms reasonably but is not perfect — especially for domain acronyms.
>
> **Fix:** Query rewriting (Advanced RAG) or hybrid BM25 + dense retrieval.  
> See [`02_interview_bank/02-advanced-rag.md`](../02_interview_bank/02-advanced-rag.md) for the upgrade path.

---
## Summary

You built a complete Naive RAG pipeline:

| Stage | What you did | Key file / class |
|---|---|---|
| **Index** | Chunked 10 docs with 3 strategies | `RecursiveCharacterTextSplitter` |
| **Embed** | Mapped chunks to vectors locally | `HuggingFaceEmbeddings` / `all-MiniLM-L6-v2` |
| **Store** | Indexed vectors in-memory | `Chroma` |
| **Retrieve** | Fetched top-3 chunks per query | `as_retriever` |
| **Generate** | Answered queries with an LLM | `RetrievalQA` + Mistral-7B |
| **Evaluate** | Measured retrieval precision@3 | Ground-truth table |
| **Fail** | Observed 2 concrete failure modes | Boundary artifact, semantic mismatch |

---

## Next Labs

| Lab | Upgrade | File |
|---|---|---|
| Lab 02 | Add BM25 hybrid search to fix semantic mismatch | `06_labs_py/02_hybrid_search.ipynb` |
| Lab 03 | Add a cross-encoder reranker to boost precision | `06_labs_py/03_reranking.ipynb` |
| Lab 04 | Evaluate with RAGAS metrics | `06_labs_py/04_ragas_evaluation.ipynb` |

## Interview bank references

- [Q1 — Naive RAG architecture](../02_interview_bank/01-naive-rag.md#q1)
- [Q2 — Limitations](../02_interview_bank/01-naive-rag.md#q2)
- [Q3 — Embedding strategies](../02_interview_bank/01-naive-rag.md#q3)
- [Q4 — Retrieval evaluation](../02_interview_bank/01-naive-rag.md#q4)
- [Q6 — Chunking strategies](../02_interview_bank/01-naive-rag.md#q6)
- [Q10 — 50-line implementation](../02_interview_bank/01-naive-rag.md#q10)